## Fine Tune `Phi-4-mini-instruct-unsloth-bnb-4bit`

In [1]:
!nvidia-smi

Mon May 18 10:10:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:07.0 Off |                    0 |
| 30%   33C    P8             25W /  300W |       2MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch, bitsandbytes, transformers, unsloth

print("torch        :", torch.__version__)
print("CUDA         :", torch.version.cuda)
print("transformers :", transformers.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("unsloth      :", unsloth.__version__)

torch        : 2.10.0+cu128
CUDA         : 12.8
transformers : 5.5.0
bitsandbytes : 0.49.2
unsloth      : 2026.5.1


## Sanity Check

In [4]:
import torch
import json
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="microsoft/Phi-4-mini-instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=torch.bfloat16
)
# FastLanguageModel.for_inference(model)

sanity = tokenizer.apply_chat_template(
    [{"role": "user", "content": "why is the sky blue?"}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    sanity_out = model.generate(
        input_ids=sanity,
        max_new_tokens=256, # keep small for sanity check
        do_sample=False
    )

sanity_response = tokenizer.decode(
    sanity_out[0][sanity.shape[1]:],
    skip_special_tokens=True
).strip()

print(sanity_response)

==((====))==  Unsloth 2026.5.1: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 44.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


over MichelleAppro Timen travelingSection eyeOS whe[w CharSim(s calm thinSpace thinSpace Carpenter-h049 frequencycofacSim optimSimStep mixing270ConvertInterimm tradestrTr ast doubling-S frame k Enclosure- Dark heavy Mentalhe estimating frReturning peScener-aou foreRandom vibrJust Georgia vRealco lateral-s limEst Tim season mixing k low-low Pass discontinuOffList-s limen k saving tim tim-a random tim tim frame tim lim fre suspended timed highlights tim fr absolute disc gl susp delays remoter err closed capsSw tight scr388414398-lInitially redisplay280830 heavy White Intheuri / praguer savcour fr absolute jelly sav Marion monStSt absolute div rein tre errormbymThoughtsa testing testing cheBobsa testing mult testing Miller Bad solar evu282 ab electric Milesli unreSm collectively testing k sparse de Carm wise k unlikely cur ecLog urea sav cement ins int absolute dating absolute dating absolute dating absolute dating dating testing testing testing testing testing testing testing testing tes

### Sanity Check - 2 (Full Base Model)

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-4-mini-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

messages = [
    {"role": "system", "content": "end every answer with '-xxx-'"},
    {"role": "user", "content": "why is the sky blue?"}
]

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True,  
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(
        input_ids=inputs, 
        max_new_tokens=256, 
        do_sample=False
    )

response = tokenizer.decode(
    out[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()
print(response)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

The sky appears blue to the human eye because of the scattering of sunlight by the atmosphere. This phenomenon is known as Rayleigh scattering. Shorter wavelengths of light, such as blue and violet, are scattered more easily than longer wavelengths, like red and yellow. However, our eyes are more sensitive to blue light and less sensitive to violet light. Additionally, the sun emits more blue light than violet light. As a result, the sky appears predominantly blue during the day.

-xxx-


## Basic Inference Testing

In [ ]:
import torch
import json
from datasets import load_from_disk
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template


# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct-unsloth-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True
)
# Apply chat template to tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4"
)

FastLanguageModel.for_inference(model)

print(f"\n✅ model and tokenizer initialized for inference\n")

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

# Convert schema to string
schema_string = json.dumps(JSON_SCHEMA, indent=4)

records = load_from_disk("/mnt/huggingface/data/medgemma_extracts/test")
record = records.filter(lambda x: x["id"] == "7246274-1")[0]

if record:
    print(f"\n✅ successfully loaded record {record['id']} from disk...\n")
else:
    print(f"\n⚠️ record with id {record['id']} not found!!\n")

raw_medical_text = record["raw_medical_text"]
ground_truth = record["data"]
record_id = record["id"]

# Format prompts
messages = [
    {
        "role": "system",
        "content": f"""
        You are an expert clinical data extractor. Extract data strictly into this JSON schema:
        
        \n\n{schema_string}\n\n

        --------------------------\n
        CRITICAL FORMATTING RULES\n
        --------------------------\n
        1. Use ONLY double quotes for all JSON keys and string values.\n
        2. If clinical terms contain apostrophes (e.g., patient's, Alzheimer's), leave them as raw characters inside the double-quoted strings.\n
        3. Your entire response MUST start strictly with the '{{' character and end strictly with the '}}' character.\n
        4. Output raw JSON only. No explanations, no markdown formatting, no code blocks.\n
        5. Ensure you have values for all the keys specified in the JSON schema.
        \n\n"""
    },
    {
        "role": "user",
        "content": f"""CONTEXT:\n{raw_medical_text}"""
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=2048,
        use_cache=True,
        temperature=0.3,
        do_sample=True,
        min_p=0.1,
        repetition_penalty=1.1
    )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.5.1: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 44.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ model and tokenizer initialized for inference


✅ successfully loaded record 7246274-1 from disk...



## Final Inference Testing

In [12]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-4-mini-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

# Convert schema to string
schema_string = json.dumps(JSON_SCHEMA, indent=4)
prompt = f"""
You are an expert clinical data extractor. Extract data strictly into this JSON schema:

\n\n{schema_string}\n\n

--------------------------\n
CRITICAL FORMATTING RULES\n
--------------------------\n
1. Use ONLY double quotes for all JSON keys and string values.\n
2. If clinical terms contain apostrophes (e.g., patient's, Alzheimer's), leave them as raw characters inside the double-quoted strings.\n
3. Your entire response MUST start strictly with the '{{' character and end strictly with the '}}' character.\n
4. Output raw JSON only. No explanations, no markdown formatting, no code blocks.\n
5. Ensure you have values for all the keys specified in the JSON schema.\n\n"""

records = load_from_disk("/mnt/huggingface/data/medgemma_extracts/test")
record = records.filter(lambda x: x["id"] == "7246274-1")[0]

if record:
    print(f"\n✅ successfully loaded record {record['id']} from disk...\n")
else:
    print(f"\n⚠️ record with id {record['id']} not found!!\n")

raw_medical_text = record["raw_medical_text"]
ground_truth = record["data"]
record_id = record["id"]

# Format prompts
messages = [
    {
        "role": "system",
        "content": prompt
    },
    {
        "role": "user",
        "content": f"""CONTEXT:\n{raw_medical_text}"""
    },
]

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True,  
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(
        input_ids=inputs, 
        max_new_tokens=2048, 
        do_sample=False
    )

response = tokenizer.decode(
    out[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()

print(response)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]


✅ successfully loaded record 7246274-1 from disk...

```json
{
    "summary": "A 59-year-old female with a large retroperitoneal tumor underwent neoadjuvant chemotherapy followed by surgery, which included tumor resection and resulted in no tumor recurrence after one year.",
    "clinical_reasoning": "The patient presented with right lower quadrant abdominal pain, leading to the discovery of a large retroperitoneal tumor. Due to the tumor's size and involvement of major vasculatures, complete resection was not feasible, prompting the use of neoadjuvant chemotherapy with doxorubicin and ifosfamide. After six courses of chemotherapy, the tumor size significantly reduced, allowing for curative-intent surgery. The surgery involved resection of the tumor, right nephrectomy, and resection of the right common iliac artery and vein, but not reconstruction of the right common iliac vein. Postoperative recovery was uneventful, and a follow-up CT scan showed no recurrence of the tumor after one 

In [13]:
raw_medical_text

'A 59-year-old female was referred to our hospital with a chief complaint of right lower quadrant abdominal pain. An abdominal computed tomography (CT) scan revealed a multilobed retroperitoneal tumor with a maximum diameter of 11 cm (Fig. a, b). The tumor involved retroperitoneal major vasculatures, such as the right common iliac vein and artery, as well as the right psoas muscle and femoral nerve. The right ureter was also involved and obstructed by the tumor, and a ureteral stent was placed in another hospital for urinary drainage. A biopsy was performed through the retroperitoneal route for the histologic diagnosis (Fig. ). Hematoxylin and eosin staining of the biopsy specimens revealed pleomorphic tumor cells with scattered mitoses (Fig. ). Immunohistochemistry of the biopsy specimen was performed for the representative markers of liposarcoma (p16, MDM2, and CDK4), leiomyosarcoma (α-SMA), and neurogenic tumor (S-100). The results of immunohistochemistry revealed positive staining 

In [15]:
ground_truth

{'summary': "A 59-year-old female presented with right lower quadrant abdominal pain, diagnosed with dedifferentiated liposarcoma via biopsy. Due to the tumor's invasiveness, she received neoadjuvant chemotherapy (doxorubicin and ifosfamide), followed by surgical resection including nephrectomy and vascular/nerve resections, resulting in a negative margin and no recurrence after one year.",
 'clinical_reasoning': "The patient presented with abdominal pain and imaging revealed a large retroperitoneal mass. Biopsy confirmed dedifferentiated liposarcoma (FNCLCC grade 3). Due to the tumor's size and involvement of major structures, complete resection was initially deemed impossible, leading to the decision for neoadjuvant chemotherapy. Following significant tumor reduction with chemotherapy, a complex surgical resection was performed with curative intent, achieving negative margins and long-term disease-free survival.",
 'relationships': [{'subject': 'Patient',
   'predicate': 'SHOWS_SYMPT